In [6]:
import sys
import os
from pathlib import Path
import pandas as pd
import numpy as np

# Determine the representative_days package location and add it to sys.path.
# This handles both running from the project root and from inside the package folder.
root = Path.cwd()
if (root / 'pre-analysis' / 'representative_days' / '__init__.py').exists():
    sys.path.insert(0, str(root / 'pre-analysis'))
elif (root / 'representative_days' / '__init__.py').exists():
    sys.path.insert(0, str(root))
elif (root.name == 'representative_days' and (root / '__init__.py').exists()):
    sys.path.insert(0, str(root.parent))
else:
    sys.path.insert(0, str(root))

from representative_days.representativeseasons_pipeline import run_representative_seasons
from representative_days.representativedays_pipeline import run_representative_days_pipeline


In [ ]:

# Define paths
base_dir = Path.cwd() 
input_dir = base_dir / 'input'
output_dir = base_dir / 'output'
input_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)

# Create sample PV data (hourly for one year)
if not (input_dir / 'pv_data.csv').exists():
    print("Creating sample PV data...")
    dates = pd.date_range('2018-01-01', periods=8760, freq='h')
    pv_data = pd.DataFrame({
        'month': dates.month,
        'day': dates.day,
        'hour': dates.hour,
        'zone': 'ZONE1',
        'value': np.random.uniform(0, 1, len(dates))  # Dummy PV output
    })
    pv_data.to_csv(input_dir / 'pv_data.csv', index=False)

# Create sample wind data
if not (input_dir / 'wind_data.csv').exists():
    print("Creating sample wind data...")
    dates = pd.date_range('2018-01-01', periods=8760, freq='h')
    wind_data = pd.DataFrame({
        'month': dates.month,
        'day': dates.day,
        'hour': dates.hour,
        'zone': 'ZONE1',
        'value': np.random.uniform(0, 1, len(dates))  # Dummy wind output
    })
    wind_data.to_csv(input_dir / 'wind_data.csv', index=False)

# Create sample load data
if not (input_dir / 'load_data.csv').exists():
    print("Creating sample load data...")
    dates = pd.date_range('2018-01-01', periods=8760, freq='h')
    load_data = pd.DataFrame({
        'month': dates.month,
        'day': dates.day,
        'hour': dates.hour,
        'zone': 'ZONE1',
        'value': 500 + 200 * np.sin(2 * np.pi * np.arange(len(dates)) / 24)  # Dummy load profile
    })
    load_data.to_csv(input_dir / 'load_data.csv', index=False)

print("Input files ready.")

# Define input files (path -> technology name)
input_files = {
    'PV': str(input_dir / 'pv_data.csv'),
    'Wind': str(input_dir / 'wind_data.csv'),
    'Load': str(input_dir / 'load_data.csv')
}

# Explicit GAMS model path for representative days pipeline
gams_main_file = str(base_dir / 'gams' / 'OptimizationModelZone.gms')

# Derive seasons (2 seasons: wet/dry, or 4: winter/spring/summer/fall)
seasons_map = run_representative_seasons(
    input_files=input_files,
    K=4,  # number of seasons
    output_path=output_dir / 'seasons.csv',
    diagnostics_dir=output_dir / 'diagnostics'
)

# Run full pipeline
results = run_representative_days_pipeline(
    seasons_map=seasons_map,
    input_files=input_files,
    output_dir=output_dir / 'representative_days',
    n_representative_days=4,
    verbose=True,
    gams_main_file=gams_main_file,
)

print("Pipeline complete!")


Input files ready.
[repr-seasons] Starting representative seasons pipeline
[repr-seasons] Loading monthly features from 3 inputs (value column: value)
[c:\Users\wb590499\Documents\Projects\EPM\pre-analysis\representative_days\input\load_data.csv] Dropping 264 Feb 29 entries for zones Eswatini, Lesotho, Malawi, Mozambique, Namibia, Zimbabwe, Zambia, Tanzania, South Africa, Angola, Botswana.
[c:\Users\wb590499\Documents\Projects\EPM\pre-analysis\representative_days\input\load_data.csv] Normalizing 'value' to [0,1] (max=3.64e+04).
[repr-seasons] Load: monthly mean range 0.086→0.094
[repr-seasons] PV: monthly mean range 0.051→0.060
[repr-seasons] Wind: monthly mean range 0.011→0.029
[repr-seasons] Built feature matrix with shape (12, 3)
[repr-seasons] Clustering months into 4 seasons (features shape: (12, 3))
[repr-seasons] Month 8 looked lonely; aligning it with month 8
[repr-seasons] Month 11 looked lonely; aligning it with month 11
[repr-seasons] Final month→season map: {1: 1, 2: 1, 3: 

c:\Users\wb590499\.conda\envs\epm_env\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


[repr-days] Step 1: preparing raw inputs
[repr-days]   prepared PV -> c:\Users\wb590499\Documents\Projects\EPM\pre-analysis\representative_days\output\representative_days\pv_data_season.csv
[repr-days]   prepared Wind -> c:\Users\wb590499\Documents\Projects\EPM\pre-analysis\representative_days\output\representative_days\wind_data_season.csv
[c:\Users\wb590499\Documents\Projects\EPM\pre-analysis\representative_days\input\load_data.csv] Dropping 264 Feb 29 entries for zones Eswatini, Lesotho, Malawi, Mozambique, Namibia, Zimbabwe, Zambia, Tanzania, South Africa, Angola, Botswana.
[c:\Users\wb590499\Documents\Projects\EPM\pre-analysis\representative_days\input\load_data.csv] Normalizing '2019' to [0,1] (max=3.64e+04).
[repr-days]   prepared Load -> c:\Users\wb590499\Documents\Projects\EPM\pre-analysis\representative_days\output\representative_days\load_data_season.csv
[repr-days] Step 2: combine tech profiles and pick representative year
                                 2019
zone     seas

'Annual capacity factor (%):'

tech,zone,PV,Wind,Load
0,Angola,0.058869,0.001404,0.050374
1,Botswana,0.055030,0.019552,0.012994
2,DRC,0.053775,0.000119,NaN
3,Eswatini,0.055373,0.007073,0.004230
4,Lesotho,0.063668,0.017986,0.003269
5,Malawi,0.056828,0.012354,0.007625
6,Mozambique,0.054283,0.017844,0.022526
7,Namibia,0.058944,0.039390,0.013417
8,South Africa,0.057082,0.031322,0.759306
9,Tanzania,0.057334,0.019394,0.036259


[repr-days] Step 3: clustering with n_clusters=20


c:\Users\wb590499\.conda\envs\epm_env\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
c:\Users\wb590499\.conda\envs\epm_env\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
c:\Users\wb590499\.conda\envs\epm_env\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


[repr-days]   special days extracted: 9
[repr-days] Step 4: building optimization input
File saved at: c:\Users\wb590499\Documents\Projects\EPM\pre-analysis\representative_days\output\representative_days\data_formatted_optim.csv
[repr-days] Step 5: running optimization for 4 representative days
File saved to: gams\bins_settings_10.csv
Launch GAMS code
Command to execute: gams c:\Users\wb590499\Documents\Projects\EPM\pre-analysis\representative_days\gams\OptimizationModelZone.gms --data c:\Users\wb590499\Documents\Projects\EPM\pre-analysis\representative_days\output\representative_days\data_formatted_optim.csv --settings c:\Users\wb590499\Documents\Projects\EPM\pre-analysis\representative_days\gams\bins_settings_10.csv --N 4
